# 03 Matching Comparison — общий benchmark reranker-моделей

Этот notebook отвечает на главный исследовательский вопрос: какой pairwise matcher лучше отличает одинаковый базовый SKU от разных товаров на одном frozen dataset. Здесь мы уже не генерируем пары и не меняем разметку, а честно сравниваем score-источники и пороги.

**Результат:** единый threshold benchmark с `dev`-калибровкой, `test`-проверкой, чистой comparison table, графиками и примерами ошибок для анализа качества.

## Оглавление

- [Мини-словарь](#мини-словарь)
- [0. Настройки запуска](#0-настройки-запуска)
- [1. Подготовка окружения](#1-подготовка-окружения)
- [2. Пути, helpers и формат таблиц](#2-пути-helpers-и-формат-таблиц)
- [3. Загружаем frozen dataset](#3-загружаем-frozen-dataset)
- [4. Единый catalog моделей](#4-единый-catalog-моделей)
- [5. Scoring helpers](#5-scoring-helpers)
- [6. Применяем модели или берём готовые scores](#6-применяем-модели-или-берём-готовые-scores)
- [7. Sales-volume веса](#7-sales-volume-веса)
- [8. Единый threshold benchmark](#8-единый-threshold-benchmark)
- [9. Чистые comparison tables](#9-чистые-comparison-tables)
- [10. Визуализации](#10-визуализации)
- [11. Примеры ошибок](#11-примеры-ошибок)
- [12. Optional archive summary](#12-optional-archive-summary)
- [13. Итоговые выводы и следующий шаг](#13-итоговые-выводы-и-следующий-шаг)

## Ход исследования

1. В одном месте объявляем frozen dataset, report paths и список моделей.
2. Для каждой модели либо берём готовый `score_csv`, либо считаем score live через matcher.
3. Собираем все scores в одну таблицу.
4. Подбираем `threshold_same` только на `dev` и проверяем выбранный порог на `test`.
5. Показываем чистые таблицы, графики и примеры ошибок.

`train` в финальных метриках не используется: threshold выбирается на `dev`, честная проверка идёт на `test`.


## Мини-словарь

`same_base_product` — бинарный таргет: `1`, если это тот же базовый товар, и `0`, если товар другой.

`score` — оценка модели для пары товаров. Чем выше score, тем больше модель верит, что товары совпадают.

`threshold_same` — порог: `score >= threshold_same` означает same-base product.

`false merge` — дорогая ошибка: разные товары склеили как один.

`false split` — менее дорогая ошибка: один и тот же базовый товар не склеили.

Главная бизнес-стратегия по умолчанию — `threshold_weighted_cost`: она минимизирует дорогие false merge с учётом объёма продаж. Для финального сравнения моделей sales-volume weights обязательны.


## 0. Настройки запуска

Менять обычно нужно только эту ячейку.

Главные ручки:

- `MY_EVAL_DATA_PATH` — frozen split, на котором сравниваем модели.
- `MY_MODEL_SPECS` — единый список моделей и источников score.
- `score_path` — готовые scores; если подходят к текущему frozen dataset, notebook использует их без пересчёта.
- `MY_CATEGORY_RUNS` — только контекст для sales-volume join; frozen benchmark остаётся multi-category.
- `model_path` или `alias` — live scoring, если готовых scores нет или они от другого split.
- `MY_ALLOW_LIVE_MODEL_SCORING` — разрешить грузить модели и считать score прямо в notebook.

Если ячейка scoring крутится дольше нескольких минут, почти всегда причина в live scoring cross-encoder/reranker-моделей. Прервите kernel и используйте `MY_MAX_EVAL_PAIRS` для быстрого smoke-run. Для финальных метрик поставьте `MY_MAX_EVAL_PAIRS = 0`.


In [ ]:
# === MY notebook settings ===

# Frozen dataset: текущий 2400+ row pair-stratified split для общего benchmark.
MY_EVAL_DATA_PATH = "research/dedup/data/training/dedup_pairs_final_pair_stratified_split.csv"
MY_SCORE_SPLITS = ["dev", "test"]
MY_REPORTS_DIR = "artifacts/reports/fine_tuning"

# Sales-volume weights. Это НЕ фильтр benchmark dataset: frozen CSV уже multi-category.
# Список нужен только для попытки подтянуть продажи из DuckDB по всем категориям сразу.
MY_CATEGORY_RUNS = ["sauces", "coconut_oil", "soap"]
MY_DUCKDB_PATH = None  # None = искать mpstats.duckdb в корне проекта
MY_ENABLE_SALES_VOLUME_JOIN = True
MY_REQUIRE_SALES_VOLUME_WEIGHTS = True  # финальный benchmark должен упасть, если веса по продажам не подтянулись

# Threshold/cost settings.
MY_FP_COST = 5.0
MY_FN_COST = 1.0
MY_PRIMARY_STRATEGY = "threshold_weighted_cost"
MY_PRIMARY_SPLIT = "test"

# Live scoring controls.
# True = можно грузить модели из registry/HF/local path и считать score.
# Если запуск идёт слишком долго, сначала прервите kernel и поставьте False или уменьшите MY_MAX_EVAL_PAIRS.
MY_ALLOW_LIVE_MODEL_SCORING = True
MY_USE_SCORE_CSV_FIRST = True
MY_MAX_EVAL_PAIRS = 0  # 120 = быстрый smoke; 0 = полный dev+test benchmark для финальных метрик
MY_RANDOM_STATE = 42

# External backup with server-side score artifacts. Нужен только если хотите подтянуть готовые CSV.
MY_EXTERNAL_REPORTS_DIR = "/Users/exoldoff/Desktop/mpstats_server_backup_20260630_041745/artifacts/reports"
MY_EXTERNAL_MODEL_DIR = "/Users/exoldoff/Desktop/mpstats_server_backup_20260630_041745/artifacts/models/dedup/exoldoff/bge-reranker-v2-m3-cross-encoder-marketplaces-rus/final"

# Единый registry сравниваемых моделей.
# source priority: score_path -> live matcher. Если score_path от другого split, он будет пропущен и при наличии модели пересчитан live.
MY_MODEL_SPECS = [
    {
        "enabled": True,
        "method": "rule_based_fuzzy",
        "family": "baseline",
        "kind": "rule_based",
        "display_name": "Rule-based fuzzy",
    },
    {
        "enabled": True,
        "method": "bi_encoder_zero_shot",
        "family": "zero-shot",
        "kind": "bi_encoder",
        "alias": "bi_encoder_e5_small",
        "display_name": "multilingual-e5-small bi-encoder",
    },
    {
        "enabled": True,
        "method": "cross_encoder_zero_shot",
        "family": "zero-shot",
        "kind": "cross_encoder",
        "alias": "cross_encoder_mmarco",
        "display_name": "mMARCO cross-encoder",
    },
    {
        "enabled": True,
        "method": "reranker_bge_v2_m3",
        "family": "zero-shot reranker",
        "kind": "cross_encoder",
        "alias": "reranker_bge_v2_m3",
        "display_name": "BGE reranker v2 m3 zero-shot",
    },
    {
        "enabled": False,
        "method": "reranker_qwen3_0_6b",
        "family": "zero-shot reranker",
        "kind": "cross_encoder",
        "alias": "reranker_qwen3_0_6b",
        "display_name": "Qwen3 reranker 0.6B zero-shot",
        "note": "disabled by default: slow live scoring on laptop CPU/MPS",
    },
    {
        "enabled": False,
        "method": "reranker_qwen3_4b",
        "family": "zero-shot reranker",
        "kind": "cross_encoder",
        "alias": "reranker_qwen3_4b",
        "display_name": "Qwen3 reranker 4B zero-shot",
        "note": "disabled by default: heavy CPU/GPU memory use",
    },
    {
        "enabled": False,
        "method": "reranker_jina_v3",
        "family": "zero-shot reranker",
        "kind": "jina_reranker",
        "alias": "reranker_jina_v3",
        "display_name": "Jina reranker v3 zero-shot",
        "note": "disabled by default: slow live scoring on laptop CPU/MPS",
    },
    {
        "enabled": True,
        "method": "ft_bge_reranker_v2_m3",
        "family": "fine-tuned",
        "kind": "cross_encoder",
        "display_name": "BGE v2 m3 marketplace fine-tune",
        "hf_model_id": "exoldoff/bge-reranker-v2-m3-cross-encoder-marketplaces-rus",
        "model_path": MY_EXTERNAL_MODEL_DIR,
        "score_path": f"{MY_EXTERNAL_REPORTS_DIR}/fine_tuning/bge_pair_stratified_scores.csv",
        "score_column": "ft_bge_reranker_v2_m3",
        "activation": "sigmoid",
        "batch_size": 32,
    },
]

# Optional appendix: старый mixed-source terminal summary из backup. По умолчанию скрыт, чтобы не смешивать split-ы.
MY_SHOW_ARCHIVE_SUMMARY = False
MY_ARCHIVE_TERMINAL_SUMMARY_PATH = f"{MY_EXTERNAL_REPORTS_DIR}/fine_tuning/all_models_terminal_summary.csv"


## 1. Подготовка окружения

Здесь только imports, project root и общие display-настройки. Модели в этой ячейке не загружаются.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import math
import sys
import time
from typing import Any

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.services.sales_filter_service import (
    DEFAULT_SALES_FILTER_GROUP_COLUMNS,
    DEFAULT_SALES_MIN_QUANTILE,
    DEFAULT_SALES_MIN_UNITS,
    filter_sales_by_quantile,
)
from research.dedup.training.sales_weights import (
    DEFAULT_SALES_COLUMN,
    SALES_COLUMN_CANDIDATES,
    coerce_sales_series,
    resolve_sales_column,
)
from research.dedup import (
    BinaryThresholdConfig,
    FusionConfig,
    RuleBasedMatcher,
    BiEncoderMatcher,
    CrossEncoderMatcher,
    JinaRerankerMatcher,
    calibrate_and_evaluate_methods,
    detect_sales_volume_columns,
    resolve_category_run,
    resolve_run_paths,
    same_base_product_target,
    write_binary_threshold_reports,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig
from research.dedup.matchers.cross_encoder import CrossEncoderConfig
from research.dedup.matchers.jina_reranker import JinaRerankerConfig
from research.dedup.model_registry import (
    CROSS_ENCODER_BACKEND,
    TRANSFORMERS_AUTO_MODEL_BACKEND,
    resolve_model_spec,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)


## 2. Пути, helpers и формат таблиц

Эта секция приводит настройки к абсолютным путям и задаёт маленькие функции для аккуратных таблиц.


In [ ]:
def _resolve_notebook_path(value: str | Path | None) -> Path | None:
    '''Превращает notebook-настройку пути в абсолютный Path относительно корня проекта.'''
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    path = Path(text).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


EVAL_DATA_PATH = _resolve_notebook_path(MY_EVAL_DATA_PATH)
REPORTS_DIR = _resolve_notebook_path(MY_REPORTS_DIR)
EXTERNAL_REPORTS_DIR = _resolve_notebook_path(MY_EXTERNAL_REPORTS_DIR)
CATEGORY_RUNS = [resolve_category_run(value) for value in MY_CATEGORY_RUNS]
CATEGORY_ALIASES = tuple(dict.fromkeys(alias for run in CATEGORY_RUNS for alias in run.category_aliases))
PROJECT_NAMES = tuple(dict.fromkeys(run.project_name for run in CATEGORY_RUNS if run.project_name))
PRODUCTS_TABLE = "mpstats_products"
SALES_VOLUME_COL = DEFAULT_SALES_COLUMN
SALES_MIN_QUANTILE = DEFAULT_SALES_MIN_QUANTILE
SALES_MIN_UNITS = DEFAULT_SALES_MIN_UNITS


def _round_frame(frame: pd.DataFrame, digits: int = 3) -> pd.DataFrame:
    '''Округляет числовые колонки перед отображением таблицы.'''
    output = frame.copy()
    for column in output.select_dtypes(include="number").columns:
        output[column] = output[column].round(digits)
    return output


def _display_clean_table(frame: pd.DataFrame, *, title: str | None = None, limit: int | None = None) -> None:
    '''Показывает компактную таблицу и явный placeholder вместо пустого вывода.'''
    output = frame.copy()
    if limit is not None:
        output = output.head(limit)
    output = _round_frame(output)
    if title:
        print(title)
    if output.empty:
        display(pd.DataFrame([{"status": "empty"}]))
    else:
        display(output)


print("Project root:", PROJECT_ROOT)
print("Eval data:", EVAL_DATA_PATH)
print("Reports dir:", REPORTS_DIR)
print("Category runs for sales join:", [run.slug for run in CATEGORY_RUNS])


## 3. Загружаем frozen dataset

На этом шаге notebook читает frozen CSV, проверяет binary target и оставляет только split-ы из `MY_SCORE_SPLITS`.

По умолчанию это `dev` и `test`: `dev` нужен для выбора threshold, `test` нужен для финальной проверки. `train` не нужен для честного сравнения моделей.


In [ ]:
def _normalise_eval_split(value: object) -> str:
    '''Приводит варианты названия validation split к единому dev/test/train контракту.'''
    text = str(value).strip().lower()
    aliases = {"validation": "dev", "valid": "dev", "val": "dev", "development": "dev"}
    return aliases.get(text, text)


def _coerce_score_splits(values: list[str]) -> list[str]:
    '''Оставляет только разрешённые split-ы для scoring и evaluation.'''
    output = [_normalise_eval_split(value) for value in values]
    return [value for value in output if value in {"train", "dev", "test"}]


def _make_pair_key(frame: pd.DataFrame) -> pd.Series:
    '''Строит стабильный ключ пары из доступных колонок frozen или score CSV.'''
    if "benchmark_pair_key" in frame.columns:
        return frame["benchmark_pair_key"].astype(str)
    if "pair_key" in frame.columns:
        return frame["pair_key"].astype(str)
    if "pair_id" in frame.columns:
        return frame["pair_id"].astype(str)
    if {"raw_record_id_a", "raw_record_id_b"}.issubset(frame.columns):
        left = frame["raw_record_id_a"].astype(str).str.strip()
        right = frame["raw_record_id_b"].astype(str).str.strip()
        return left + " || " + right
    return pd.Series([f"row::{idx}" for idx in frame.index], index=frame.index, dtype="object")


def _with_benchmark_pair_key(frame: pd.DataFrame) -> pd.DataFrame:
    '''Добавляет benchmark_pair_key, чтобы безопасно join-ить scores с frozen dataset.'''
    output = frame.copy()
    output["benchmark_pair_key"] = _make_pair_key(output)
    return output


def load_frozen_dataset(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Читает frozen CSV, строит binary target и возвращает пары вместе со статусом загрузки.'''
    if path is None or not path.exists():
        return pd.DataFrame(), pd.DataFrame([{
            "status": "missing_eval_data_file",
            "path": str(path),
            "rows_total": 0,
            "rows_with_target": 0,
        }])
    raw = pd.read_csv(path)
    target = same_base_product_target(raw)
    labeled = raw[target.notna()].copy()
    labeled["same_base_product"] = target[target.notna()].astype(int).to_numpy()
    if "split" not in labeled.columns:
        raise ValueError("Frozen dataset must contain split=train/dev/test")
    labeled["eval_split"] = labeled["split"].map(_normalise_eval_split)
    labeled = _with_benchmark_pair_key(labeled)
    status = pd.DataFrame([{
        "status": "ready" if not labeled.empty else "empty",
        "path": str(path),
        "rows_total": len(raw),
        "rows_with_target": len(labeled),
        "ignored_without_target": int(len(raw) - len(labeled)),
        "splits": labeled["eval_split"].value_counts().to_dict(),
        "positive_pairs": int(labeled["same_base_product"].eq(1).sum()),
        "negative_pairs": int(labeled["same_base_product"].eq(0).sum()),
    }])
    return labeled.reset_index(drop=True), status


score_splits = _coerce_score_splits(MY_SCORE_SPLITS)
labeled_pairs, dataset_status = load_frozen_dataset(EVAL_DATA_PATH)
benchmark_pairs = labeled_pairs[labeled_pairs["eval_split"].isin(score_splits)].copy().reset_index(drop=True)

_display_clean_table(dataset_status, title="Frozen dataset status")
_display_clean_table(
    benchmark_pairs.groupby(["eval_split", "same_base_product"], as_index=False).size().rename(columns={"size": "rows"}),
    title="Rows used for scoring/evaluation",
)


## 4. Единый catalog моделей

Здесь notebook просто показывает, какие модели включены, откуда будут браться scores и что будет fallback-ом.

Это весь список сравнения в одном месте. Ниже уже не будет отдельных “добавим ещё одну модель” секций.


In [ ]:
def _normalise_model_config(raw: dict[str, Any]) -> dict[str, Any]:
    '''Нормализует одну запись model registry из первой code-ячейки.'''
    config = dict(raw)
    config["enabled"] = bool(config.get("enabled", True))
    config["method"] = str(config.get("method") or "").strip()
    config["family"] = str(config.get("family") or "other")
    config["kind"] = str(config.get("kind") or "").strip()
    config["display_name"] = str(config.get("display_name") or config["method"])
    config["score_path"] = _resolve_notebook_path(config.get("score_path"))
    config["model_path"] = _resolve_notebook_path(config.get("model_path"))
    return config


model_configs = [_normalise_model_config(item) for item in MY_MODEL_SPECS]
model_catalog = pd.DataFrame([
    {
        "enabled": cfg["enabled"],
        "method": cfg["method"],
        "family": cfg["family"],
        "kind": cfg["kind"],
        "score_csv": cfg["score_path"].name if cfg.get("score_path") else "",
        "live_source": str(cfg.get("alias") or cfg.get("hf_model_id") or cfg.get("model_path") or ""),
        "note": cfg.get("note", ""),
    }
    for cfg in model_configs
])

_display_clean_table(model_catalog, title="Model catalog")


## 5. Scoring helpers

Один общий механизм для всех моделей:

- сначала применяет общий cap `MY_MAX_EVAL_PAIRS`, если он задан;
- затем пытается взять совместимый `score_csv`;
- если CSV нет или он от другого dataset/split, можно пересчитать score live;
- если модель отключена или недоступна, она не ломает notebook, а попадает в status table.


In [ ]:
class ScoreSourceMismatch(ValueError):
    '''Сигнализирует, что готовый score CSV не соответствует текущему frozen dataset.'''
    pass


def _infer_score_column(frame: pd.DataFrame, configured: object) -> str:
    '''Находит числовую колонку score в готовом score CSV.'''
    configured_text = str(configured or "").strip()
    if configured_text and configured_text in frame.columns:
        return configured_text
    candidates = [
        column
        for column in frame.columns
        if pd.api.types.is_numeric_dtype(frame[column])
        and str(column) not in {"same_base_product", "labels", "label", "sku_a", "sku_b"}
        and str(column).startswith(("ft_", "zs_", "reranker_", "score_"))
    ]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(f"Cannot infer score column. configured={configured_text!r}, candidates={candidates}")


def _score_frame_from_csv(config: dict[str, Any], score_path: Path, expected_pairs: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    '''Проверяет совместимость score CSV с frozen dataset и возвращает scores в общем формате.'''
    raw = pd.read_csv(score_path)
    if "eval_split" not in raw.columns:
        if "split" not in raw.columns:
            raise ScoreSourceMismatch("score_csv has no split/eval_split column")
        raw["eval_split"] = raw["split"].map(_normalise_eval_split)
    raw = _with_benchmark_pair_key(raw)
    score_column = _infer_score_column(raw, config.get("score_column"))
    raw[score_column] = pd.to_numeric(raw[score_column], errors="coerce")
    raw = raw[raw["eval_split"].isin(score_splits)].copy()
    raw = raw.dropna(subset=[score_column])
    raw = raw.drop_duplicates(subset=["benchmark_pair_key"], keep="last")

    expected = expected_pairs[["benchmark_pair_key"]].drop_duplicates()
    joined = expected.merge(raw[["benchmark_pair_key", score_column]], on="benchmark_pair_key", how="left")
    missing_count = int(joined[score_column].isna().sum())
    coverage = 1.0 - (missing_count / len(expected)) if len(expected) else 0.0
    if missing_count:
        raise ScoreSourceMismatch(
            f"score_csv does not match current frozen scope: coverage={coverage:.1%}, missing_pairs={missing_count}"
        )

    output = expected_pairs.copy()
    output = output.merge(raw[["benchmark_pair_key", score_column]], on="benchmark_pair_key", how="left")
    output["method"] = config["method"]
    output["score"] = pd.to_numeric(output[score_column], errors="coerce")
    output["benchmark_source"] = f"score_csv:{score_path.name}"
    status = {
        "method": config["method"],
        "status": "ready_score_csv",
        "source": str(score_path),
        "score_column": score_column,
        "pairs": len(output),
        "coverage": round(coverage, 4),
    }
    return output.reset_index(drop=True), status


def _sample_for_live_scoring(frame: pd.DataFrame, max_pairs: int) -> pd.DataFrame:
    '''Применяет smoke-limit к benchmark pairs перед live scoring.'''
    if max_pairs <= 0 or frame.empty or len(frame) <= max_pairs:
        return frame.copy().reset_index(drop=True)
    sampled = (
        frame.assign(_order=frame.groupby(["eval_split", "same_base_product"]).cumcount())
        .sort_values(["_order", "eval_split", "same_base_product", "benchmark_pair_key"])
        .head(max_pairs)
        .drop(columns=["_order"])
        .reset_index(drop=True)
    )
    return sampled


def _matcher_from_config(config: dict[str, Any]) -> Any:
    '''Создаёт matcher нужного типа из notebook model config.'''
    kind = config.get("kind")
    if kind == "rule_based":
        return RuleBasedMatcher()
    if kind == "bi_encoder":
        alias = str(config.get("alias") or config.get("model_name") or "bi_encoder_e5_small")
        spec = resolve_model_spec(alias)
        return BiEncoderMatcher(BiEncoderConfig(
            model_name=spec.model_name,
            model_backend=spec.backend,
            batch_size=int(config.get("batch_size") or spec.batch_size or 32),
            text_prefix=spec.text_prefix,
        ))
    if kind == "cross_encoder":
        model_name = str(config.get("model_path") or config.get("model_name") or "")
        spec = None
        if not model_name:
            alias = str(config.get("alias") or config.get("hf_model_id") or "cross_encoder_mmarco")
            spec = resolve_model_spec(alias, backend=CROSS_ENCODER_BACKEND)
            model_name = spec.model_name
        return CrossEncoderMatcher(CrossEncoderConfig(
            model_name=model_name,
            method_name=config["method"],
            batch_size=int(config.get("batch_size") or (spec.batch_size if spec else 16) or 16),
            device=config.get("device") if "device" in config else (spec.device if spec else None),
            trust_remote_code=bool(config.get("trust_remote_code", spec.trust_remote_code if spec else False)),
            prompts=config.get("prompts") if "prompts" in config else (spec.prompts if spec else None),
            default_prompt_name=config.get("default_prompt_name") if "default_prompt_name" in config else (spec.default_prompt_name if spec else None),
            activation=config.get("activation"),
            fusion=FusionConfig(
                threshold_high=(spec.fusion_threshold_high if spec and spec.fusion_threshold_high is not None else 0.0),
                threshold_low=(spec.fusion_threshold_low if spec and spec.fusion_threshold_low is not None else -5.0),
            ),
        ))
    if kind == "jina_reranker":
        alias = str(config.get("alias") or config.get("model_name") or "reranker_jina_v3")
        spec = resolve_model_spec(alias, backend=TRANSFORMERS_AUTO_MODEL_BACKEND)
        return JinaRerankerMatcher(JinaRerankerConfig(
            model_name=spec.model_name,
            method_name=config["method"],
            documents_per_query=int(config.get("documents_per_query") or spec.documents_per_query or 8),
            trust_remote_code=bool(config.get("trust_remote_code", spec.trust_remote_code)),
            fusion=FusionConfig(
                threshold_high=(spec.fusion_threshold_high if spec.fusion_threshold_high is not None else 0.5),
                threshold_low=(spec.fusion_threshold_low if spec.fusion_threshold_low is not None else 0.2),
            ),
        ))
    raise ValueError(f"Unsupported model kind: {kind!r}")


def _score_matcher(config: dict[str, Any], matcher: Any, pairs: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    '''Считает live scores одной моделью и возвращает их в benchmark-контракте.'''
    if pairs.empty:
        return pd.DataFrame(), {"method": config["method"], "status": "skipped_empty_scope", "pairs": 0}
    started = time.perf_counter()
    row_objects = [row for _, row in pairs.iterrows()]
    if hasattr(matcher, "score_batch"):
        scores = matcher.score_batch(row_objects)
    else:
        scores = [matcher.score(row) for row in row_objects]
    elapsed = time.perf_counter() - started
    output = pairs.copy()
    output["method"] = config["method"]
    output["score"] = pd.to_numeric(pd.Series(scores), errors="coerce").to_numpy()
    output["benchmark_source"] = "live_model"
    status = getattr(matcher, "status", lambda: None)()
    message = getattr(status, "message", "") if status is not None else ""
    ready_scores = int(output["score"].notna().sum())
    return output.reset_index(drop=True), {
        "method": config["method"],
        "status": "ready_live_model" if ready_scores else "failed_no_numeric_scores",
        "source": str(config.get("alias") or config.get("model_path") or config.get("hf_model_id") or config.get("kind")),
        "pairs": len(output),
        "numeric_scores": ready_scores,
        "seconds": round(elapsed, 2),
        "seconds_per_pair": round(elapsed / len(output), 4) if len(output) else 0,
        "message": message,
    }


## 6. Применяем модели или берём готовые scores

Эта ячейка создаёт единую `all_score_frames` таблицу. В status table видно, откуда пришёл score для каждой модели.

Если `score_csv` не совпадает с текущим frozen dataset, notebook не смешивает его в сравнение. При разрешённом live scoring он попробует пересчитать модель на текущем dataset.


In [ ]:
all_score_frames: list[pd.DataFrame] = []
score_status_rows: list[dict[str, Any]] = []
evaluation_pairs = _sample_for_live_scoring(benchmark_pairs, int(MY_MAX_EVAL_PAIRS or 0))

_display_clean_table(
    evaluation_pairs.groupby(["eval_split", "same_base_product"], as_index=False).size().rename(columns={"size": "rows"})
    if not evaluation_pairs.empty else pd.DataFrame(),
    title=f"Effective pair scope after MY_MAX_EVAL_PAIRS={MY_MAX_EVAL_PAIRS}",
)

if evaluation_pairs.empty:
    score_status_rows.append({"method": "all", "status": "skipped_empty_benchmark_scope"})
else:
    for config in model_configs:
        method = config.get("method") or "unknown"
        if not config.get("enabled", True):
            score_status_rows.append({
                "method": method,
                "status": "skipped_disabled",
                "source": config.get("note", "disabled in MY_MODEL_SPECS"),
            })
            continue

        frame: pd.DataFrame | None = None
        csv_error = ""
        score_path = config.get("score_path")
        if MY_USE_SCORE_CSV_FIRST and score_path is not None:
            if score_path.exists():
                try:
                    frame, status = _score_frame_from_csv(config, score_path, evaluation_pairs)
                    all_score_frames.append(frame)
                    score_status_rows.append({**status, "family": config.get("family"), "display_name": config.get("display_name")})
                    continue
                except Exception as exc:
                    csv_error = str(exc)
            else:
                csv_error = f"score_csv_missing: {score_path}"

        if not MY_ALLOW_LIVE_MODEL_SCORING:
            score_status_rows.append({
                "method": method,
                "status": "skipped_live_scoring_disabled",
                "source": str(score_path or config.get("alias") or config.get("model_path") or ""),
                "csv_error": csv_error,
                "family": config.get("family"),
                "display_name": config.get("display_name"),
            })
            continue

        try:
            matcher = _matcher_from_config(config)
            frame, status = _score_matcher(config, matcher, evaluation_pairs)
            if not frame.empty and frame["score"].notna().any():
                all_score_frames.append(frame)
            score_status_rows.append({
                **status,
                "csv_error": csv_error,
                "family": config.get("family"),
                "display_name": config.get("display_name"),
            })
        except Exception as exc:
            score_status_rows.append({
                "method": method,
                "status": "failed_live_model",
                "source": str(config.get("alias") or config.get("model_path") or config.get("hf_model_id") or ""),
                "error": str(exc),
                "csv_error": csv_error,
                "family": config.get("family"),
                "display_name": config.get("display_name"),
            })

score_status = pd.DataFrame(score_status_rows)
all_scores = pd.concat(all_score_frames, ignore_index=True) if all_score_frames else pd.DataFrame()

_display_clean_table(score_status, title="Score source status")
_display_clean_table(
    all_scores.groupby(["method", "benchmark_source", "eval_split"], as_index=False).size().rename(columns={"size": "rows"})
    if not all_scores.empty else pd.DataFrame(),
    title="Scores collected for benchmark",
)


## 7. Sales-volume веса

Если в score table уже есть `sales_volume_a/b`, notebook использует их. Иначе попробует подтянуть продажи из DuckDB по `raw_record_id_a/b`.

`MY_CATEGORY_RUNS` здесь не фильтрует benchmark. Он только задаёт, по каким project/category aliases искать продажи в DuckDB: по умолчанию `sauces`, `coconut_oil` и `soap` вместе.

Финальный режим требует weighted metrics: при `MY_REQUIRE_SALES_VOLUME_WEIGHTS=True` notebook остановится, если продажи не подтянулись.


In [ ]:
def _marketplace_key(value: object) -> str:
    '''Нормализует marketplace в формат raw_record_id, совместимый с research pairs.'''
    if value is None:
        return "unknown_marketplace"
    try:
        if bool(value != value):
            return "unknown_marketplace"
    except TypeError:
        return "unknown_marketplace"
    text = str(value).casefold().strip()
    return " ".join(text.split()) if text else "unknown_marketplace"


def _resolve_duckdb_path(project_root: Path) -> Path | None:
    '''Находит существующий DuckDB-куб для подтягивания объёма продаж.'''
    explicit_path = Path(MY_DUCKDB_PATH).expanduser() if MY_DUCKDB_PATH else None
    candidates = [explicit_path, project_root / "mpstats.duckdb"]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    return None


def _load_sales_volume_lookup() -> tuple[pd.DataFrame, str | None]:
    '''Читает продажи из DuckDB и собирает lookup raw_record_id -> sales_volume.'''
    if not MY_ENABLE_SALES_VOLUME_JOIN:
        return pd.DataFrame(), "weighted metrics disabled: MY_ENABLE_SALES_VOLUME_JOIN=False"
    if importlib.util.find_spec("duckdb") is None:
        return pd.DataFrame(), "weighted metrics disabled: duckdb package is not available"
    db_path = _resolve_duckdb_path(PROJECT_ROOT)
    if db_path is None:
        return pd.DataFrame(), "weighted metrics disabled: mpstats.duckdb was not found; set MY_DUCKDB_PATH"

    import duckdb

    with duckdb.connect(str(db_path), read_only=True) as con:
        tables = con.execute("SHOW TABLES").fetchdf().iloc[:, 0].astype(str).tolist()
        if PRODUCTS_TABLE not in tables:
            return pd.DataFrame(), f"weighted metrics disabled: table {PRODUCTS_TABLE!r} not found in {db_path}"
        columns = set(con.execute(f"DESCRIBE {PRODUCTS_TABLE}").fetchdf()["column_name"].astype(str))
        resolved_sales_volume_col = resolve_sales_column(columns, SALES_VOLUME_COL)
        missing = sorted({"Маркетплейс", "Артикул"} - columns)
        if resolved_sales_volume_col is None:
            missing.append(f"sales volume column, expected one of: {list(SALES_COLUMN_CANDIDATES)}")
        if missing:
            return pd.DataFrame(), f"weighted metrics disabled: missing columns in {PRODUCTS_TABLE}: {missing}"

        where_clauses: list[str] = []
        params: list[object] = []
        if "Категория" in columns and CATEGORY_ALIASES:
            placeholders = ", ".join(["?"] * len(CATEGORY_ALIASES))
            where_clauses.append(f'"Категория" IN ({placeholders})')
            params.extend(CATEGORY_ALIASES)
        if PROJECT_NAMES:
            if "__project_name" not in columns:
                return pd.DataFrame(), "weighted metrics disabled: project filter is configured but __project_name is missing"
            placeholders = ", ".join(["?"] * len(PROJECT_NAMES))
            where_clauses.append(f'"__project_name" IN ({placeholders})')
            params.extend(PROJECT_NAMES)

        group_columns = [column for column in DEFAULT_SALES_FILTER_GROUP_COLUMNS if column in columns]
        select_columns = list(dict.fromkeys(["Маркетплейс", "Артикул", resolved_sales_volume_col, *group_columns]))
        select_sql = ", ".join(f'"{column}"' for column in select_columns)
        where_sql = f" WHERE {' AND '.join(where_clauses)}" if where_clauses else ""
        raw = con.execute(f"SELECT {select_sql} FROM {PRODUCTS_TABLE}{where_sql}", params).fetchdf()

    raw = filter_sales_by_quantile(
        raw,
        sales_column=resolved_sales_volume_col,
        quantile=SALES_MIN_QUANTILE,
        min_sales=SALES_MIN_UNITS,
        group_columns=DEFAULT_SALES_FILTER_GROUP_COLUMNS,
    ).reset_index(drop=True)
    if raw.empty:
        return pd.DataFrame(), "weighted metrics disabled: sales lookup is empty after filtering"
    raw["raw_record_id"] = raw["Маркетплейс"].map(_marketplace_key) + "::" + raw["Артикул"].astype(str).str.strip()
    raw["sales_volume"] = coerce_sales_series(raw[resolved_sales_volume_col]).clip(lower=0)
    lookup = raw.groupby("raw_record_id", as_index=False)["sales_volume"].sum()
    return lookup, None


def _attach_sales_volume(frame: pd.DataFrame) -> tuple[pd.DataFrame, str | None]:
    '''Добавляет sales_volume_a/b к scored pairs, если benchmark CSV ещё не содержит веса.'''
    if frame.empty or detect_sales_volume_columns(frame) is not None:
        return frame.copy(), None
    lookup, warning = _load_sales_volume_lookup()
    if lookup.empty:
        return frame.copy(), warning
    output = frame.copy()
    volume_map = lookup.set_index("raw_record_id")["sales_volume"]
    output["sales_volume_a"] = output.get("raw_record_id_a", pd.Series(dtype="object")).astype(str).map(volume_map)
    output["sales_volume_b"] = output.get("raw_record_id_b", pd.Series(dtype="object")).astype(str).map(volume_map)
    matched_rows = int(output[["sales_volume_a", "sales_volume_b"]].notna().any(axis=1).sum())
    if matched_rows == 0:
        return frame.copy(), "weighted metrics disabled: sales lookup did not match benchmark raw_record_id values"
    return output, None


all_scores_weighted, sales_warning = _attach_sales_volume(all_scores)
if sales_warning:
    display(pd.DataFrame([{"sales_volume_status": sales_warning}]))
    if MY_REQUIRE_SALES_VOLUME_WEIGHTS:
        raise RuntimeError(f"{sales_warning}. Финальный benchmark требует sales-volume weights; проверьте MY_DUCKDB_PATH/MY_CATEGORY_RUNS и колонку продаж.")
else:
    detected_volume_cols = detect_sales_volume_columns(all_scores_weighted)
    display(pd.DataFrame([{
        "sales_volume_status": "available" if detected_volume_cols else "not_available",
        "sales_volume_columns": detected_volume_cols,
    }]))


## 8. Единый threshold benchmark

Здесь все модели проходят одну и ту же процедуру:

- threshold выбирается только на `dev`;
- `test` используется только для проверки;
- все strategies считаются одной функцией;
- CSV сохраняются в один reports dir.


In [ ]:
threshold_config = BinaryThresholdConfig(fp_cost=float(MY_FP_COST), fn_cost=float(MY_FN_COST))

if all_scores_weighted.empty:
    binary_results = {
        "summary": pd.DataFrame(),
        "predictions": pd.DataFrame(),
        "threshold_grid_dev": pd.DataFrame(),
        "weights_available": False,
        "weight_source": None,
        "weight_warning": "no scores available",
    }
    report_paths = {}
else:
    binary_results = calibrate_and_evaluate_methods(
        all_scores_weighted,
        config=threshold_config,
        sales_volume_cols=detect_sales_volume_columns(all_scores_weighted),
    )
    if MY_REQUIRE_SALES_VOLUME_WEIGHTS and not bool(binary_results.get("weights_available", False)):
        raise RuntimeError(f"Weighted benchmark is required but unavailable: {binary_results.get('weight_warning')}")
    report_paths = write_binary_threshold_reports(binary_results, REPORTS_DIR)

binary_threshold_summary = binary_results.get("summary", pd.DataFrame())
binary_threshold_predictions = binary_results.get("predictions", pd.DataFrame())
threshold_grid_dev = binary_results.get("threshold_grid_dev", pd.DataFrame())

# Добавляем metadata о source/family для чистых comparison tables.
metadata_cols = ["method", "family", "display_name"]
model_metadata = score_status[[col for col in metadata_cols if col in score_status.columns]].drop_duplicates("method") if not score_status.empty else pd.DataFrame(columns=metadata_cols)
if isinstance(binary_threshold_summary, pd.DataFrame) and not binary_threshold_summary.empty:
    binary_threshold_summary = binary_threshold_summary.merge(model_metadata, on="method", how="left")
    all_models_threshold_summary_path = REPORTS_DIR / "all_models_threshold_summary.csv"
    binary_threshold_summary.to_csv(all_models_threshold_summary_path, index=False)
else:
    all_models_threshold_summary_path = None

saved_paths = {key: str(value) for key, value in report_paths.items()}
if all_models_threshold_summary_path is not None:
    saved_paths["all_models_threshold_summary"] = str(all_models_threshold_summary_path)

display(pd.DataFrame([{
    "methods_evaluated": int(binary_threshold_summary["method"].nunique()) if isinstance(binary_threshold_summary, pd.DataFrame) and not binary_threshold_summary.empty else 0,
    "weights_available": bool(binary_results.get("weights_available", False)),
    "weight_source": binary_results.get("weight_source"),
    "weight_warning": binary_results.get("weight_warning"),
    "saved_reports": saved_paths,
}]))


## 9. Чистые comparison tables

Основная таблица ниже — одна строка на модель для выбранной стратегии `MY_PRIMARY_STRATEGY` на `MY_PRIMARY_SPLIT`.

Сортировка: сначала меньше `weighted_total_cost`, если weighted metrics доступны; иначе меньше обычный `cost`, затем больше F1.


In [ ]:
def _primary_metric_column(frame: pd.DataFrame) -> str:
    '''Выбирает главную cost-колонку для сортировки comparison table.'''
    if "weighted_total_cost" in frame.columns and pd.to_numeric(frame["weighted_total_cost"], errors="coerce").notna().any():
        return "weighted_total_cost"
    return "cost"


def _effective_strategy(summary: pd.DataFrame, *, split: str, preferred: str) -> str:
    '''Возвращает доступную threshold strategy с безопасным fallback для weighted режимов.'''
    if summary.empty:
        return preferred
    split_rows = summary[summary["split"].astype(str).eq(split)]
    available = set(split_rows["threshold_strategy"].astype(str))
    if preferred in available:
        return preferred
    if preferred == "threshold_weighted_cost" and "threshold_cost_sensitive" in available:
        return "threshold_cost_sensitive"
    if preferred == "threshold_max_weighted_f1" and "threshold_max_f1" in available:
        return "threshold_max_f1"
    return sorted(available)[0] if available else preferred


def _comparison_table(summary: pd.DataFrame, *, split: str, strategy: str) -> pd.DataFrame:
    '''Собирает финальную таблицу сравнения моделей для одного split и strategy.'''
    if summary.empty:
        return pd.DataFrame()
    frame = summary[
        summary["split"].astype(str).eq(split)
        & summary["threshold_strategy"].astype(str).eq(strategy)
    ].copy()
    if frame.empty and strategy == "threshold_weighted_cost":
        frame = summary[
            summary["split"].astype(str).eq(split)
            & summary["threshold_strategy"].astype(str).eq("threshold_cost_sensitive")
        ].copy()
    if frame.empty:
        return frame
    cost_col = _primary_metric_column(frame)
    sort_cols = [cost_col, "false_merge_count", "false_split_count", "f1"]
    ascending = [True, True, True, False]
    frame = frame.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
    frame.insert(0, "rank", range(1, len(frame) + 1))
    keep_cols = [
        "rank",
        "method",
        "family",
        "display_name",
        "threshold_strategy",
        "threshold_same",
        "precision",
        "recall",
        "f1",
        "accuracy",
        "false_merge_count",
        "false_split_count",
        "cost",
        "weighted_f1",
        "weighted_total_cost",
        "weight_source",
    ]
    return frame[[col for col in keep_cols if col in frame.columns]]


PRIMARY_STRATEGY_EFFECTIVE = _effective_strategy(binary_threshold_summary, split=MY_PRIMARY_SPLIT, preferred=MY_PRIMARY_STRATEGY)
terminal_summary = _comparison_table(binary_threshold_summary, split=MY_PRIMARY_SPLIT, strategy=PRIMARY_STRATEGY_EFFECTIVE)
if not terminal_summary.empty:
    terminal_summary_path = REPORTS_DIR / "all_models_terminal_summary.csv"
    terminal_summary.to_csv(terminal_summary_path, index=False)
else:
    terminal_summary_path = None

_display_clean_table(terminal_summary, title=f"Main comparison: {MY_PRIMARY_SPLIT} / {PRIMARY_STRATEGY_EFFECTIVE}")

all_test_strategies = binary_threshold_summary[binary_threshold_summary.get("split", pd.Series(dtype=str)).astype(str).eq(MY_PRIMARY_SPLIT)].copy() if not binary_threshold_summary.empty else pd.DataFrame()
if not all_test_strategies.empty:
    compact_cols = [
        "method",
        "threshold_strategy",
        "precision",
        "recall",
        "f1",
        "false_merge_count",
        "false_split_count",
        "cost",
        "weighted_total_cost",
    ]
    all_test_strategies = all_test_strategies[[col for col in compact_cols if col in all_test_strategies.columns]]
    all_test_strategies = all_test_strategies.sort_values(["method", "threshold_strategy"])

_display_clean_table(all_test_strategies, title=f"All threshold strategies on {MY_PRIMARY_SPLIT}", limit=80)

if terminal_summary_path is not None:
    print("Terminal summary saved:", terminal_summary_path)


## 10. Визуализации

Графики намеренно используют только основную comparison table, чтобы не было каши из `dev/test` и четырёх threshold strategies одновременно.


In [ ]:
try:
    # In a real Jupyter kernel matplotlib renders inline. In terminal smoke-tests, use Agg so plt.show() does not block.
    if "ipykernel" not in sys.modules:
        import matplotlib
        matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
except Exception as exc:
    display(pd.DataFrame([{"visualization_status": "matplotlib_unavailable", "error": str(exc)}]))
else:
    if terminal_summary.empty:
        display(pd.DataFrame([{"visualization_status": "no_terminal_summary"}]))
    else:
        plt.rcParams.update({
            "figure.facecolor": "white",
            "axes.grid": True,
            "grid.alpha": 0.25,
            "axes.spines.top": False,
            "axes.spines.right": False,
        })

        plot_frame = terminal_summary.copy()
        plot_frame["label"] = plot_frame["method"].astype(str)
        for col in ["f1", "precision", "recall", "cost", "weighted_total_cost", "false_merge_count", "false_split_count"]:
            if col in plot_frame.columns:
                plot_frame[col] = pd.to_numeric(plot_frame[col], errors="coerce")

        cost_col = _primary_metric_column(plot_frame)
        cost_title = "Weighted total cost" if cost_col == "weighted_total_cost" else "Cost"
        height = max(4.5, 0.42 * len(plot_frame) + 1.2)
        ordered_cost = plot_frame.sort_values(cost_col, ascending=True)
        ordered_quality = plot_frame.sort_values("f1", ascending=True)

        fig, axes = plt.subplots(1, 2, figsize=(16, height), constrained_layout=True)
        axes[0].barh(ordered_cost["label"], ordered_cost[cost_col], color="#f59e0b", alpha=0.88)
        axes[0].set_title(f"{MY_PRIMARY_SPLIT}: {cost_title} lower is better")
        axes[0].set_xlabel(cost_title)
        for idx, value in enumerate(ordered_cost[cost_col].fillna(0)):
            axes[0].text(float(value) + max(float(ordered_cost[cost_col].fillna(0).max()) * 0.01, 0.5), idx, f"{float(value):.0f}", va="center", fontsize=8)

        axes[1].barh(ordered_quality["label"], ordered_quality["f1"], color="#2563eb", alpha=0.88, label="F1")
        axes[1].scatter(ordered_quality["precision"], ordered_quality["label"], color="#16a34a", s=28, label="precision")
        axes[1].scatter(ordered_quality["recall"], ordered_quality["label"], color="#dc2626", s=28, label="recall")
        axes[1].set_xlim(0, 1.03)
        axes[1].set_title(f"{MY_PRIMARY_SPLIT}: quality metrics")
        axes[1].set_xlabel("higher is better")
        axes[1].legend(loc="lower right")
        plt.show()

        error_frame = plot_frame.sort_values(["false_merge_count", "false_split_count", "f1"], ascending=[True, True, False])
        fig, ax = plt.subplots(figsize=(12, height), constrained_layout=True)
        ax.barh(error_frame["label"], error_frame["false_split_count"], color="#60a5fa", label="false splits")
        ax.barh(error_frame["label"], error_frame["false_merge_count"], left=error_frame["false_split_count"], color="#ef4444", label="false merges")
        ax.set_title(f"{MY_PRIMARY_SPLIT}: error counts for {PRIMARY_STRATEGY_EFFECTIVE}")
        ax.set_xlabel("pairs")
        ax.legend(loc="lower right")
        plt.show()

        if not binary_threshold_predictions.empty:
            selected_predictions = binary_threshold_predictions[
                binary_threshold_predictions["split"].astype(str).eq(MY_PRIMARY_SPLIT)
                & binary_threshold_predictions["threshold_strategy"].astype(str).eq(PRIMARY_STRATEGY_EFFECTIVE)
            ].copy()
            top_methods = terminal_summary.head(6)["method"].astype(str).tolist()
            selected_predictions = selected_predictions[selected_predictions["method"].isin(top_methods)]
            if not selected_predictions.empty:
                fig, axes = plt.subplots(len(top_methods), 1, figsize=(12, max(3.0, 2.0 * len(top_methods))), constrained_layout=True)
                if len(top_methods) == 1:
                    axes = [axes]
                for ax, method in zip(axes, top_methods):
                    method_rows = selected_predictions[selected_predictions["method"].eq(method)].copy()
                    same_scores = pd.to_numeric(method_rows.loc[method_rows["same_base_product"].eq(1), "score"], errors="coerce").dropna()
                    diff_scores = pd.to_numeric(method_rows.loc[method_rows["same_base_product"].eq(0), "score"], errors="coerce").dropna()
                    threshold = pd.to_numeric(method_rows["threshold_same"], errors="coerce").dropna()
                    bins = 30
                    ax.hist(diff_scores, bins=bins, alpha=0.55, label="different", color="#ef4444")
                    ax.hist(same_scores, bins=bins, alpha=0.55, label="same", color="#2563eb")
                    if not threshold.empty:
                        ax.axvline(float(threshold.iloc[0]), color="#111827", linestyle="--", linewidth=1.2, label="threshold")
                    ax.set_title(method)
                    ax.legend(loc="upper right")
                plt.show()


## 11. Примеры ошибок

Эти таблицы нужны не для красоты, а для следующего шага: руками посмотреть, почему модель ошибается.

- `false_merge_examples` — самые опасные ошибки.
- `false_split_examples` — missed duplicates, которые можно использовать для следующего улучшения разметки или threshold.


In [ ]:
def _error_examples(predictions: pd.DataFrame, *, error_column: str, limit: int = 20) -> pd.DataFrame:
    '''Выбирает наиболее важные false merge или false split примеры для ручного разбора.'''
    if predictions.empty or error_column not in predictions.columns:
        return pd.DataFrame()
    rows = predictions[
        predictions["split"].astype(str).eq(MY_PRIMARY_SPLIT)
        & predictions["threshold_strategy"].astype(str).eq(PRIMARY_STRATEGY_EFFECTIVE)
        & predictions[error_column].astype(bool)
    ].copy()
    if rows.empty:
        return rows
    if "pair_importance" not in rows.columns:
        rows["pair_importance"] = 1.0
    rows["score_margin"] = (pd.to_numeric(rows["score"], errors="coerce") - pd.to_numeric(rows["threshold_same"], errors="coerce")).abs()
    keep_cols = [
        "method",
        "score",
        "threshold_same",
        "score_margin",
        "pair_importance",
        "label",
        "brand_a",
        "brand_b",
        "title_a",
        "title_b",
        "raw_record_id_a",
        "raw_record_id_b",
    ]
    rows = rows.sort_values(["pair_importance", "score_margin"], ascending=[False, True])
    return rows[[col for col in keep_cols if col in rows.columns]].head(limit)


false_merge_examples = _error_examples(binary_threshold_predictions, error_column="false_merge", limit=20)
false_split_examples = _error_examples(binary_threshold_predictions, error_column="false_split", limit=20)

_display_clean_table(false_merge_examples, title="False merge examples: different products merged as same", limit=20)
_display_clean_table(false_split_examples, title="False split examples: same products missed", limit=20)


## 12. Optional archive summary

Этот блок выключен по умолчанию. Он нужен только если хочется посмотреть старый server-side terminal summary, где часть моделей могла считаться на другом split.

В основной benchmark выше эти строки не смешиваются, чтобы сравнение оставалось честным.


In [ ]:
archive_summary_path = _resolve_notebook_path(MY_ARCHIVE_TERMINAL_SUMMARY_PATH)
if not MY_SHOW_ARCHIVE_SUMMARY:
    print("Archive summary hidden. Set MY_SHOW_ARCHIVE_SUMMARY=True in the first code cell if needed.")
elif archive_summary_path is None or not archive_summary_path.exists():
    display(pd.DataFrame([{"archive_status": "missing", "path": str(archive_summary_path)}]))
else:
    archive_summary = pd.read_csv(archive_summary_path)
    keep_cols = [
        "method",
        "run_type",
        "threshold_strategy",
        "precision",
        "recall",
        "f1",
        "false_merge_count",
        "false_split_count",
        "cost",
        "weighted_total_cost",
        "source_file",
    ]
    archive_summary = archive_summary[[col for col in keep_cols if col in archive_summary.columns]]
    archive_summary = archive_summary.sort_values(
        ["weighted_total_cost" if "weighted_total_cost" in archive_summary.columns else "cost", "false_merge_count", "f1"],
        ascending=[True, True, False],
    )
    _display_clean_table(archive_summary, title="Archive terminal summary (read-only, may mix score scopes)", limit=40)


## 13. Итоговые выводы и следующий шаг

После запуска сверху вниз смотри в первую очередь:

1. `Score source status` — все ли модели реально попали в сравнение.
2. `Main comparison` — какой метод выигрывает по выбранной strategy.
3. График error counts — есть ли false merges у выбранной модели.
4. `False merge examples` и `False split examples` — что стоит проверить руками.

Для downstream `04_fusion_pack_grouping.ipynb` обычно берём метод и threshold strategy из `Main comparison`, но решение принимаем только по `dev`-selected threshold и финальному `test` readout.
